# 08 — XGBoost v2

Tuned XGBoost with 30-config random search and fold-level early stopping (no log1p transformation — raw target is better suited to this bimodal distribution).

Outputs:
- `outputs/oof_XGBoostV2.npy`
- `outputs/test_local_pred_XGBoostV2.npy`
- `outputs/xgb_v2_best_params.json`
- `outputs/xgb_v2_search_log.csv`

## 1. Setup & Data

In [1]:
import warnings, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_SPLITS = 5

OUT_DIR = Path('../outputs'); OUT_DIR.mkdir(exist_ok=True, parents=True)
print('OUT_DIR =', OUT_DIR.resolve())

OUT_DIR = /Users/bashkal/Desktop/ML/ML-Final/Internship/outputs


In [2]:
train_df = pd.read_parquet(OUT_DIR / 'train_local.parquet')
test_local_df = pd.read_parquet(OUT_DIR / 'test_local.parquet')
TARGET = 'blocked_days_Q1_2026'; ID = 'id'
y = train_df[TARGET].astype(float).values
X = train_df.drop(columns=[TARGET, ID]).reset_index(drop=True)
X_test_local = test_local_df.drop(columns=[TARGET, ID]).reset_index(drop=True)
y_test_local = test_local_df[TARGET].astype(float).values

cat_all = X.select_dtypes(exclude='number').columns.tolist()
card    = {c: X[c].nunique(dropna=False) for c in cat_all}
cat_high = [c for c, n in card.items() if n > 15]
cat_low  = [c for c, n in card.items() if n <= 15]
num_cols = X.select_dtypes(include='number').columns.tolist()
print(f'Train: {X.shape} | Test local: {X_test_local.shape}')
print(f'num={len(num_cols)}  cat_low={cat_low}  cat_high={cat_high}')


Train: (29008, 145) | Test local: (7253, 145)
num=145  cat_low=[]  cat_high=[]


## 2. K-Fold Target Encoder (Module-7 custom transformer)

In [3]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, n_splits=5, smoothing=20.0, random_state=42):
        self.cols = cols; self.n_splits = n_splits
        self.smoothing = smoothing; self.random_state = random_state

    def _smap(self, x, y):
        st = pd.DataFrame({'c': x, 'y': y}).groupby('c')['y'].agg(['mean', 'count'])
        return ((st['count'] * st['mean'] + self.smoothing * self.global_mean_)
                / (st['count'] + self.smoothing)).to_dict()

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.global_mean_ = float(y.mean())
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return self

    def transform(self, X):
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = (X[c].astype(str).fillna('__nan__').map(self.maps_[c])
                       .fillna(self.global_mean_).astype('float32'))
        return Xo

    def fit_transform(self, X, y=None, **kw):
        y = np.asarray(y, dtype=float); self.global_mean_ = float(y.mean())
        Xo = X.copy()
        for c in self.cols: Xo[c] = np.full(len(X), self.global_mean_, dtype='float32')
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in kf.split(X):
            for c in self.cols:
                m = self._smap(X[c].astype(str).fillna('__nan__').iloc[tr], y[tr])
                Xo.iloc[va, Xo.columns.get_loc(c)] = (
                    X[c].astype(str).fillna('__nan__').iloc[va].map(m)
                       .fillna(self.global_mean_).astype('float32').values)
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return Xo

    def get_feature_names_out(self, input_features=None):
        return np.array(input_features if input_features is not None else self.cols)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
        ('low', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
            ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_low),
        ('high', Pipeline([
            ('te', KFoldTargetEncoder(cols=cat_high, n_splits=5, smoothing=20, random_state=RANDOM_STATE)),
        ]), cat_high),
    ])

## 3. Fold-Level Early-Stopping Evaluator
Each candidate hyperparameter config is evaluated like this:
1. Run a 5-fold CV with the *same* outer split as our other notebooks.
2. **Inside each fold:** carve a 10% inner-validation slice from the training portion. Fit XGBoost with `early_stopping_rounds=50` watching that inner slice. Use the resulting model to predict the outer-val slice → fold MSE.
3. Return mean ± std over folds.
This makes `n_estimators` adaptive: each fold picks the best iteration on its own.

In [4]:
outer_kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
outer_splits = list(outer_kf.split(X))   # cache so all configs see the same folds

def evaluate_config(params, return_oof=False, return_models=False, verbose=False):
    """Run 5-fold CV with fold-level early stopping. Returns mean MSE, std, best_iters."""
    fold_mses, best_iters, models = [], [], []
    oof = np.zeros(len(y)) if return_oof else None

    for fold, (tr, va) in enumerate(outer_splits):
        pp = make_preprocessor()
        X_tr_full = pp.fit_transform(X.iloc[tr], y[tr])
        X_va      = pp.transform(X.iloc[va])

        X_tr, X_in, y_tr, y_in = train_test_split(
            X_tr_full, y[tr], test_size=0.10, random_state=RANDOM_STATE + fold,
        )

        model = XGBRegressor(
            **params,
            objective='reg:squarederror',
            tree_method='hist',
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            early_stopping_rounds=50,
            eval_metric='rmse',
        )
        model.fit(X_tr, y_tr, eval_set=[(X_in, y_in)], verbose=False)
        best_iter = model.best_iteration

        pred = np.clip(model.predict(X_va, iteration_range=(0, best_iter + 1)), 0, 90)
        mse  = mean_squared_error(y[va], pred)

        fold_mses.append(mse); best_iters.append(best_iter)
        if return_oof: oof[va] = pred
        if return_models: models.append((model, pp, best_iter))
        if verbose: print(f'    fold {fold+1}: MSE={mse:.3f}, best_iter={best_iter}')

    return {
        'mse_mean': float(np.mean(fold_mses)),
        'mse_std':  float(np.std(fold_mses)),
        'best_iters': best_iters,
        'fold_mses': fold_mses,
        'oof': oof,
        'models': models,
    }

## 4. Sanity-Check Config (Sensible Defaults)
Confirms the pipeline works and gives us a sane baseline before any search.

In [5]:
default_cfg = dict(
    n_estimators=3000,           # large; early stopping picks the real count
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=4,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
)
t0 = time.time()
res0 = evaluate_config(default_cfg, verbose=True)
print(f"\nDefault XGB | MSE = {res0['mse_mean']:.3f} ± {res0['mse_std']:.3f} "
      f"| iters = {res0['best_iters']} | {time.time()-t0:.1f}s")

    fold 1: MSE=329.369, best_iter=351
    fold 2: MSE=348.378, best_iter=476
    fold 3: MSE=328.592, best_iter=267
    fold 4: MSE=334.536, best_iter=219
    fold 5: MSE=327.513, best_iter=321

Default XGB | MSE = 333.678 ± 7.736 | iters = [351, 476, 267, 219, 321] | 11.5s


## 5. Random Search — 60 configurations
Search space focused on the ranges that matter for tabular boosting. Learning rates are sampled log-uniform but biased toward the practical 0.03-0.08 sweet spot.

In [6]:
rng = np.random.default_rng(RANDOM_STATE)

def sample_config():
    return dict(
        n_estimators     = 3000,
        learning_rate    = float(np.exp(rng.uniform(np.log(0.02), np.log(0.10)))),
        max_depth        = int(rng.integers(4, 9)),
        min_child_weight = int(rng.integers(1, 12)),
        subsample        = float(rng.uniform(0.65, 0.95)),
        colsample_bytree = float(rng.uniform(0.65, 0.95)),
        gamma            = float(rng.uniform(0.0, 0.4)),
        reg_alpha        = float(np.exp(rng.uniform(np.log(1e-3), np.log(0.5)))),
        reg_lambda       = float(np.exp(rng.uniform(np.log(0.5),  np.log(5.0)))),
    )

N_ITER = 30
log_rows = []
best_score = res0['mse_mean']
best_cfg   = default_cfg.copy()

t_search = time.time()
for i in range(N_ITER):
    cfg = sample_config()
    t0 = time.time()
    res = evaluate_config(cfg)
    elapsed = time.time() - t0
    log_rows.append({**cfg, 'mse_mean': res['mse_mean'], 'mse_std': res['mse_std'],
                     'avg_best_iter': int(np.mean(res['best_iters'])), 'seconds': elapsed})
    flag = ''
    if res['mse_mean'] < best_score:
        best_score = res['mse_mean']
        best_cfg   = cfg.copy()
        flag = '  ←  NEW BEST'
    print(f"[{i+1:2d}/{N_ITER}] MSE = {res['mse_mean']:7.3f} ± {res['mse_std']:5.3f} "
          f"| iters≈{int(np.mean(res['best_iters'])):4d} | lr={cfg['learning_rate']:.4f} "
          f"depth={cfg['max_depth']} | {elapsed:5.1f}s{flag}")

print(f'\nSearch finished in {(time.time()-t_search)/60:.1f} min')
print(f'Best CV MSE: {best_score:.3f}')
print('Best config:', best_cfg)

log_df = pd.DataFrame(log_rows).sort_values('mse_mean').reset_index(drop=True)
log_df.to_csv(OUT_DIR / 'xgb_v2_search_log.csv', index=False)
with open(OUT_DIR / 'xgb_v2_best_params.json', 'w') as f:
    json.dump(best_cfg, f, indent=2)
log_df.head(10)

[ 1/30] MSE = 335.919 ± 10.590 | iters≈ 182 | lr=0.0695 depth=7 |   8.2s
[ 2/30] MSE = 334.440 ± 7.626 | iters≈ 232 | lr=0.0709 depth=6 |   8.7s
[ 3/30] MSE = 331.511 ± 8.753 | iters≈ 504 | lr=0.0408 depth=6 |  13.6s  ←  NEW BEST
[ 4/30] MSE = 338.261 ± 8.574 | iters≈ 925 | lr=0.0354 depth=4 |  16.1s
[ 5/30] MSE = 332.429 ± 9.118 | iters≈ 481 | lr=0.0256 depth=7 |  17.5s
[ 6/30] MSE = 330.563 ± 9.177 | iters≈ 642 | lr=0.0271 depth=6 |  17.4s  ←  NEW BEST
[ 7/30] MSE = 338.753 ± 8.229 | iters≈ 566 | lr=0.0617 depth=4 |  10.4s
[ 8/30] MSE = 329.946 ± 9.646 | iters≈ 490 | lr=0.0250 depth=8 |  22.7s  ←  NEW BEST
[ 9/30] MSE = 335.032 ± 10.721 | iters≈ 397 | lr=0.0419 depth=6 |  10.8s
[10/30] MSE = 337.423 ± 9.056 | iters≈ 254 | lr=0.0685 depth=6 |   7.7s
[11/30] MSE = 334.229 ± 8.389 | iters≈ 782 | lr=0.0282 depth=5 |  15.8s
[12/30] MSE = 338.961 ± 8.600 | iters≈ 586 | lr=0.0580 depth=4 |  10.6s
[13/30] MSE = 338.414 ± 8.742 | iters≈1313 | lr=0.0207 depth=4 |  21.5s
[14/30] MSE = 337.384 ±

,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,mse_mean,mse_std,avg_best_iter,seconds
0,3000,0.025045,8,3,0.652209,0.886077,0.265940,0.080024,3.017860,329.945832,9.646397,490,22.657705
1,3000,0.027131,6,2,0.792711,0.718073,0.267926,0.015131,3.401326,330.563206,9.177037,642,17.414699
2,3000,0.046100,6,4,0.881604,0.848498,0.149463,0.001799,2.790999,330.675563,8.073475,447,12.474962
3,3000,0.025212,8,1,0.718897,0.689547,0.271063,0.002132,1.604353,330.685871,8.037957,381,18.027033
4,3000,0.040828,6,3,0.816375,0.669145,0.331052,0.050681,2.864559,331.511474,8.752758,504,13.646478
5,3000,0.025637,7,8,0.873429,0.940253,0.130330,0.009997,1.474096,332.429214,9.118445,481,17.541880
6,3000,0.034009,7,2,0.681021,0.826293,0.068237,0.313958,1.905597,332.612514,10.456249,343,13.906630
7,3000,0.023819,7,8,0.734370,0.847827,0.290798,0.118729,0.640783,332.979755,9.089415,510,16.775554
8,3000,0.043771,7,6,0.931348,0.821518,0.189396,0.005255,1.072850,333.769232,9.160828,290,10.518999
9,3000,0.046234,8,5,0.656484,0.897888,0.358464,0.002391,1.790631,333.879681,10.403226,263,12.723596


## 6. Final 5-Fold OOF with Best Config
Re-evaluate the winner to also extract OOF predictions and the per-fold trained models (for the test-set inference).

In [7]:
final_res = evaluate_config(best_cfg, return_oof=True, return_models=True, verbose=True)
print(f"\nXGBoost v2 | OOF MSE = {final_res['mse_mean']:.3f} ± {final_res['mse_std']:.3f}")
np.save(OUT_DIR / 'oof_XGBoostV2.npy', final_res['oof'])

    fold 1: MSE=329.139, best_iter=511
    fold 2: MSE=348.570, best_iter=404
    fold 3: MSE=326.031, best_iter=346
    fold 4: MSE=324.699, best_iter=546
    fold 5: MSE=321.290, best_iter=645

XGBoost v2 | OOF MSE = 329.946 ± 9.646


## 7. Test-Set Prediction — Average the 5 Fold-Models
Each fold's model used early stopping → averaging them is more robust than refitting on full data with a fixed iter count.

In [8]:
# Predict on local test set using fold models
test_local_preds_v2 = np.zeros(len(X_test_local))
for model, pp, best_iter in final_res['models']:
    X_tl = pp.transform(X_test_local)
    p = np.clip(model.predict(X_tl, iteration_range=(0, best_iter + 1)), 0, 90)
    test_local_preds_v2 += p
test_local_preds_v2 /= len(final_res['models'])

test_local_mse_v2 = mean_squared_error(y_test_local, test_local_preds_v2)
print(f'XGBoost v2 | Local Test MSE = {test_local_mse_v2:.3f}')
np.save(OUT_DIR / 'test_local_pred_XGBoostV2.npy', test_local_preds_v2)
print('Saved test_local_pred_XGBoostV2.npy')

XGBoost v2 | Local Test MSE = 325.490
Saved test_local_pred_XGBoostV2.npy


In [9]:
print(log_df.head(10).to_string(index=False))


 n_estimators  learning_rate  max_depth  min_child_weight  subsample  colsample_bytree    gamma  reg_alpha  reg_lambda   mse_mean   mse_std  avg_best_iter   seconds
         3000       0.025045          8                 3   0.652209          0.886077 0.265940   0.080024    3.017860 329.945832  9.646397            490 22.657705
         3000       0.027131          6                 2   0.792711          0.718073 0.267926   0.015131    3.401326 330.563206  9.177037            642 17.414699
         3000       0.046100          6                 4   0.881604          0.848498 0.149463   0.001799    2.790999 330.675563  8.073475            447 12.474962
         3000       0.025212          8                 1   0.718897          0.689547 0.271063   0.002132    1.604353 330.685871  8.037957            381 18.027033
         3000       0.040828          6                 3   0.816375          0.669145 0.331052   0.050681    2.864559 331.511474  8.752758            504 13.646478
         3

## 8. Top-20 Search Configs (for the paper)

In [10]:
show_cols = ['mse_mean', 'mse_std', 'learning_rate', 'max_depth', 'min_child_weight',
             'subsample', 'colsample_bytree', 'gamma', 'reg_alpha', 'reg_lambda',
             'avg_best_iter', 'seconds']
log_df.head(20)[show_cols]

,mse_mean,mse_std,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,avg_best_iter,seconds
0,329.945832,9.646397,0.025045,8,3,0.652209,0.886077,0.265940,0.080024,3.017860,490,22.657705
1,330.563206,9.177037,0.027131,6,2,0.792711,0.718073,0.267926,0.015131,3.401326,642,17.414699
2,330.675563,8.073475,0.046100,6,4,0.881604,0.848498,0.149463,0.001799,2.790999,447,12.474962
3,330.685871,8.037957,0.025212,8,1,0.718897,0.689547,0.271063,0.002132,1.604353,381,18.027033
4,331.511474,8.752758,0.040828,6,3,0.816375,0.669145,0.331052,0.050681,2.864559,504,13.646478
5,332.429214,9.118445,0.025637,7,8,0.873429,0.940253,0.130330,0.009997,1.474096,481,17.541880
6,332.612514,10.456249,0.034009,7,2,0.681021,0.826293,0.068237,0.313958,1.905597,343,13.906630
7,332.979755,9.089415,0.023819,7,8,0.734370,0.847827,0.290798,0.118729,0.640783,510,16.775554
8,333.769232,9.160828,0.043771,7,6,0.931348,0.821518,0.189396,0.005255,1.072850,290,10.518999
9,333.879681,10.403226,0.046234,8,5,0.656484,0.897888,0.358464,0.002391,1.790631,263,12.723596


## Summary
- XGBoost tuned with 30-config random search, fold-level early stopping (no log1p).
- Best config saved to `xgb_v2_best_params.json`.
- OOF and local test predictions saved for blending.